# Bilge · Turkish AI Assistant
4-bit QLoRA fine-tuning of Qwen2.5-1.5B-Instruct on cleaned Turkish instruction data, measured comparison with the base model and Bilge's chat interface.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!nvidia-smi

In [ ]:
GITHUB_USER = "mahdildarr"
!git clone https://github.com/{GITHUB_USER}/bilge.git
%cd bilge
!pip install -q -e ".[gpu,demo,hub]"

## 1. Data cleaning + QLoRA training
With the default settings (5,000 examples, 1 epoch) this takes roughly 1–1.5 hours on a T4. For a quick trial add `--set train_samples=1000`. The number of examples removed by each cleaning filter and the perplexity before and after training are saved in `metrics.json`.

In [ ]:
!python -m bilge.train --config configs/qwen2.5-1.5b.yaml

In [ ]:
import json
m = json.load(open("runs/bilge-1.5b-lora/metrics.json"))
print("Data filters:", m["data_filters"])
print("Perplexity:", m["perplexity_before"], "->", m["perplexity_after"])

## 2. Base model ↔ Bilge

In [ ]:
!python -m bilge.compare --adapter runs/bilge-1.5b-lora/adapter
from IPython.display import Markdown, display
display(Markdown(open("runs/comparison/comparison.md", encoding="utf-8").read()))

## 3. Knowledge retention (optional)
Set `MC_DATASET` to a Turkish multiple-choice dataset on the Hugging Face Hub and adjust the column names.

In [ ]:
MC_DATASET = ""  # e.g. a Turkish MMLU-style dataset
if MC_DATASET:
    !python -m bilge.mc_eval --adapter runs/bilge-1.5b-lora/adapter --dataset {MC_DATASET} --split test --limit 1000

## 4. Chat interface
The generated `gradio.live` link is publicly accessible for 72 hours.

In [ ]:
!python app.py --adapter runs/bilge-1.5b-lora/adapter --share

## Save results
Copy the outputs to Google Drive and update the results table in the README.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/bilge
!cp -r runs /content/drive/MyDrive/bilge/